# 23. Pick a Reference Timestamp for Population Counting

<span style="font-family: 'Courier New', monospace;">

*AI-generated draft (Claude, Anthropic) — for review. All parameters and figures are derived from version-controlled scripts and data.*

The goal is to find **one fixed second** where the camera is zoomed in on the Mushroom vent with worms clearly visible. That timestamp will be used to extract one consistent frame per video across the full dataset.

**Instructions:**
1. Run **Cell 1** — it pre-extracts frames every 5 seconds across the full video (~2 min)
2. Run **Cell 2** — drag the slider to scrub through the video and find the best frame
3. Note the timestamp shown above the frame — plug it into `24_count_timeseries.ipynb`

</span>

In [ ]:
import subprocess
from pathlib import Path
import matplotlib.image as mpimg

# ── Configuration ──────────────────────────────────────────────────
REFERENCE_VIDEO = Path(
    "/home/jovyan/ooi/san_data/RS03ASHS-PN03B-06-CAMHDA301/"
    "2024/10/01/CAMHDA301-20241001T031500.mp4"
)
STEP_SEC         = 5     # extract one frame every N seconds
VIDEO_DURATION_SEC = 1500  # ~25 min; raise if video is longer
CACHE_DIR        = Path("./timestamp_cache")
# ──────────────────────────────────────────────────────────────────

CACHE_DIR.mkdir(exist_ok=True)
timestamps = list(range(0, VIDEO_DURATION_SEC, STEP_SEC))

print(f"Pre-extracting {len(timestamps)} frames (every {STEP_SEC}s) ...")
print("Skipping frames already cached.\n")

for i, t in enumerate(timestamps):
    out = CACHE_DIR / f"frame_{t:05d}.png"
    if out.exists():
        continue
    subprocess.run(
        ["ffmpeg", "-y", "-ss", str(t), "-i", str(REFERENCE_VIDEO),
         "-frames:v", "1", "-q:v", "2", str(out)],
        capture_output=True, timeout=30,
    )
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(timestamps)} done ...")

# Load all frames into memory for instant slider response
frame_cache = {}
for t in timestamps:
    p = CACHE_DIR / f"frame_{t:05d}.png"
    if p.exists():
        frame_cache[t] = mpimg.imread(str(p))

print(f"\nCached {len(frame_cache)} frames. Run Cell 2 to open the slider.")

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

sorted_times = sorted(frame_cache.keys())

slider = widgets.SelectionSlider(
    options=[(f"{t//60}m {t%60:02d}s  (t={t}s)", t) for t in sorted_times],
    description="Time:",
    layout=widgets.Layout(width="900px"),
    style={"description_width": "50px"},
)

out = widgets.Output()

def show_frame(change):
    t = change["new"]
    with out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(frame_cache[t])
        m, s = divmod(t, 60)
        ax.set_title(
            f"t = {t}s  ({m}m {s:02d}s)  —  {REFERENCE_VIDEO.name}",
            fontsize=14, fontweight="bold",
        )
        ax.axis("off")
        plt.tight_layout()
        plt.show()
        print(f"  ✓ Current timestamp: {t}s  ({m}m {s:02d}s)")
        print(f"    Plug this into 24_count_timeseries.ipynb as REFERENCE_TIMESTAMP = {t}")

slider.observe(show_frame, names="value")
display(widgets.VBox([slider, out]))
show_frame({"new": sorted_times[0]})